[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_01/3_stats.ipynb)

# Day 1: Statistics, A/B testing and product (easy)

**How to work:** Answer in two layers: the number (code) and one plain sentence a product manager understands. Name the method and its assumptions before computing.

**Today (25 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | What is a typical tip? | easy | 6 |
| Q2 | How expensive is a taxi ride? | easy | 6 |
| Q3 | One player, 49,854 rounds | easy | 6 |
| Q4 | Longer bills, shallower bills? | easy | 7 |

**Handbook:** [Statistics, A/B testing and product](https://mahsa-agz.github.io/ds-handbook/#stats), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

All real datasets (see DATASETS.md for sources). The setup cell loads them:

- `tips`: 244 restaurant bills (`total_bill`, `tip`, `sex`, `smoker`, `day`, `time`, `size`).
- `taxis`: 6,433 New York taxi trips from March 2019 (`pickup`, `distance`, `fare`, `tip`, `total`, `payment`, `pickup_borough` ...).
- `cookie`: Cookie Cats mobile game A/B test, 90,189 players. `version` is `gate_30` (control) or `gate_40` (treatment: the first gate moved from level 30 to 40); `sum_gamerounds` = rounds played in the first week; `retention_1`, `retention_7` = came back 1 / 7 days after install (bool).
- `penguins`: 344 penguins (`species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`; some NaN).

Classic probability questions need no data: solve them on paper first, then check with a simulation (`rng = np.random.default_rng(0)`).

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import numpy as np
import pandas as pd
from scipy import stats as st
pd.set_option("display.width", 120)

tips = pd.read_csv(data_path("tips.csv", "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"))
taxis = pd.read_csv(data_path("taxis.csv", "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"), parse_dates=["pickup", "dropoff"])
cookie = pd.read_csv(data_path("cookie_cats.csv", "https://raw.githubusercontent.com/robguilarr/ab_testing_cookie_cats/main/datasets/cookie_cats.csv"))
penguins = pd.read_csv(data_path("penguins.csv", "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"))
for _name in ['tips', 'taxis', 'cookie', 'penguins']:
    print(_name, globals()[_name].shape)

---
### Q1. What is a typical tip?

*easy, about 6 min*

The restaurant owner asks: *what does a typical table spend, and what tip rate do people leave?*

Using `tips`:
1. Print the mean, median and standard deviation of `total_bill`, and its skewness.
2. Make `tip_pct = 100 * tip / total_bill` and print its mean, median and maximum.
3. Which single number would you report as the *typical* bill, and why?

Example of the output shape: `bill mean 19.79 ...`.

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# tips is loaded in the setup

<details><summary><b>Hint 1</b></summary>

Signal: 'typical' value of a money column. Money is right skewed (a few big bills), so compare the mean and the median before you choose. Topic: center and skew.

</details>

<details><summary><b>Hint 2</b></summary>

1. `tips.total_bill.agg(['mean', 'median', 'std', 'skew'])`. 2. Compute tip_pct as a new Series (do not overwrite `tips`). 3. Mean > median and skew > 0 means a long right tail: report the median as typical and the mean when you need totals.

</details>

<details><summary><b>Solution</b></summary>

```python
bill = tips["total_bill"]
print(f"bill mean {bill.mean():.2f}  median {bill.median():.2f}  sd {bill.std():.2f}  skew {bill.skew():.2f}")
tip_pct = 100 * tips["tip"] / tips["total_bill"]
print(f"tip% mean {tip_pct.mean():.2f}  median {tip_pct.median():.2f}  max {tip_pct.max():.2f}")
row = tips.loc[tip_pct.idxmax()]
print(f"max tip% row: bill {row.total_bill:.2f}, tip {row.tip:.2f}")

# Output:
# bill mean 19.79  median 17.80  sd 8.90  skew 1.13
# tip% mean 16.08  median 15.48  max 71.03
# max tip% row: bill 7.25, tip 5.15
```

**Why:** The mean is pulled up by a few large bills (skew 1.13 > 0, so mean 19.79 > median 17.80). The median is the bill of the 'middle' table and is robust to the tail, so it is the better *typical* value. The mean is still the right number for totals (revenue = mean x number of tables). The 71% tip is a \$5.15 tip on a \$7.25 bill: a ratio with a small denominator explodes, which is why the median tip rate (15.48%) is safer than the mean (16.08%). pandas `std()` uses `n - 1` (sample sd); numpy `np.std` uses `n` by default.

**Say it to a PM:** A typical table spends about \$17.80 (half spend less, half more) and leaves about 15.5% as a tip. The average bill is higher, \$19.79, because a few large parties pull it up.

**Common mistakes:** Reporting only the mean for a skewed money metric. Averaging ratios without looking at tiny denominators. Mixing `np.std` (divides by n) with `pandas.std` (divides by n - 1).

**Learn more:** [stats-descriptive](https://mahsa-agz.github.io/ds-handbook/#stats-descriptive), [cheat-pandas](https://mahsa-agz.github.io/ds-handbook/#cheat-pandas)

</details>

---
### Q2. How expensive is a taxi ride?

*easy, about 6 min*

A ride-hailing PM wants one slide on taxi fares. Using `taxis.fare`:
1. Print the 50th, 90th and 99th percentiles and the IQR (Q3 minus Q1).
2. Flag outliers with the 1.5 x IQR rule (above `Q3 + 1.5 * IQR`). Print the cut-off and the share of trips above it.
3. Should the flagged trips be deleted before analysis? Answer in one sentence.

Example: if Q1 = 4 and Q3 = 10, the IQR is 6 and the upper fence is 19.

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# taxis is loaded in the setup

<details><summary><b>Hint 1</b></summary>

Signal: 'one slide' on a long-tailed price. Report percentiles (p50, p90, p99), not mean +- sd. Topic: percentiles, IQR and outlier rules.

</details>

<details><summary><b>Hint 2</b></summary>

1. `q = taxis.fare.quantile([0.25, 0.5, 0.75, 0.9, 0.99])`. 2. `iqr = q[0.75] - q[0.25]`, `fence = q[0.75] + 1.5 * iqr`. 3. `(taxis.fare > fence).mean()` is the share. 4. Look at what the flagged trips are (long distance, airports) before you call them errors.

</details>

<details><summary><b>Solution</b></summary>

```python
fare = taxis["fare"]
q = fare.quantile([0.25, 0.5, 0.75, 0.9, 0.99])
iqr = q[0.75] - q[0.25]
fence = q[0.75] + 1.5 * iqr
print(f"p50 {q[0.5]:.2f}  p90 {q[0.9]:.2f}  p99 {q[0.99]:.2f}  IQR {iqr:.2f}")
print(f"upper fence {fence:.2f}  share above {(fare > fence).mean():.3f}")
print(f"median distance (miles): flagged {taxis.loc[fare > fence, 'distance'].median():.2f}, "
      f"others {taxis.loc[fare <= fence, 'distance'].median():.2f}")

# Output:
# p50 9.50  p90 26.00  p99 52.00  IQR 8.50
# upper fence 27.75  share above 0.092
# median distance (miles): flagged 11.41, others 1.50
```

**Why:** Fares are bounded below and have a long right tail, so percentiles describe them better than the mean and sd. The 1.5 x IQR rule is a quick screen, not a verdict: the flagged trips are much longer, so they are real long rides (airports, outer boroughs), not data errors. Delete only impossible values (negative fares, zero distance with a huge fare); keep real extremes and use robust statistics or a log scale.

**Say it to a PM:** Half of the rides cost \$9.50 or less, 9 in 10 cost \$26 or less, and only 1 in 100 costs more than \$52. About 9% of rides are long trips (median 11.4 miles) that cost far more; they are a real segment, not bad data.

**Common mistakes:** Deleting every point outside the fence (you lose the airport business). Using mean +- 3 sd on a skewed metric. Forgetting that pandas `quantile` interpolates between values by default.

**Learn more:** [stats-descriptive](https://mahsa-agz.github.io/ds-handbook/#stats-descriptive)

</details>

---
### Q3. One player, 49,854 rounds

*easy, about 6 min*

In the Cookie Cats data, `sum_gamerounds` is the number of rounds a player played in the first week.

1. Print the mean, median, standard deviation and maximum, and the share of players with 0 rounds.
2. Remove only the single player with the maximum and print the mean and sd again.
3. Which statistic changed a lot, which barely changed, and why?

Example: for the values 1, 2, 3, 1000 the mean is 251.5 but the median is 2.5.

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# cookie is loaded in the setup

<details><summary><b>Hint 1</b></summary>

Signal: an engagement count with a huge maximum. Compare a non-robust statistic (mean, sd) with a robust one (median) before and after removing one point. Topic: robustness and outliers.

</details>

<details><summary><b>Hint 2</b></summary>

1. `g = cookie.sum_gamerounds`; print `g.mean(), g.median(), g.std(), g.max(), (g == 0).mean()`. 2. `h = g[g < g.max()]` drops the one top player (check it is only one). 3. Compare.

</details>

<details><summary><b>Solution</b></summary>

```python
g = cookie["sum_gamerounds"]
print(f"mean {g.mean():.2f}  median {g.median():.0f}  sd {g.std():.1f}  max {g.max()}  zero share {(g == 0).mean():.3f}")
h = g[g < g.max()]                       # drops exactly one player
print(f"rows removed {len(g) - len(h)}")
print(f"without max: mean {h.mean():.2f}  median {h.median():.0f}  sd {h.std():.1f}  max {h.max()}")

# Output:
# mean 51.87  median 16  sd 195.1  max 49854  zero share 0.044
# rows removed 1
# without max: mean 51.32  median 16  sd 102.7  max 2961
```

**Why:** The sd squares the distances from the mean, so one player at 49,854 rounds (about 5 rounds a minute, around the clock for a whole week, so likely a bot or a logging bug) dominates it: removing 1 of 90,189 rows nearly halves the sd. The mean moves only a little and the median does not move at all. In an A/B test this matters: a huge sd makes the t-test weak, and one extreme user can flip the sign of a mean difference. Fixes: investigate and remove clear errors, cap (winsorize) at a high percentile such as p99, or compare medians or use a log scale.

**Say it to a PM:** A typical player played 16 rounds in the first week, and 4.4% never played at all. The average (about 52) is three times the typical value because a few heavy players pull it up, and one account with 49,854 rounds looks like a bot, so we report the median and clean that account before any A/B comparison.

**Common mistakes:** Quoting the mean and sd of a heavy-tailed metric without checking the maximum. Removing outliers with a rule chosen after seeing the A/B result (that is p-hacking). Forgetting the 4.4% of players who never played a round.

**Learn more:** [stats-descriptive](https://mahsa-agz.github.io/ds-handbook/#stats-descriptive), [stats-ab-pitfalls](https://mahsa-agz.github.io/ds-handbook/#stats-ab-pitfalls)

</details>

---
### Q4. Longer bills, shallower bills?

*easy, about 7 min*

A biologist plots `bill_length_mm` against `bill_depth_mm` for all `penguins` and says: *longer bills are shallower, the correlation is negative.*

1. Print the Pearson correlation for all penguins together (drop rows with missing values).
2. Print it again inside each species.
3. Explain the difference in one sentence. What is this effect called?

**Then say it to a PM:** one or two plain sentences with the number and what it means for the product (no jargon).

In [ ]:
# penguins is loaded in the setup

<details><summary><b>Hint 1</b></summary>

Signal: one overall number versus the same number inside groups. Topic: correlation and a lurking grouping variable (Simpson's paradox).

</details>

<details><summary><b>Hint 2</b></summary>

1. `d = penguins.dropna(subset=['bill_length_mm', 'bill_depth_mm'])`. 2. `d.bill_length_mm.corr(d.bill_depth_mm)`. 3. Loop over `d.groupby('species')` and compute the same correlation per group.

</details>

<details><summary><b>Solution</b></summary>

```python
d = penguins.dropna(subset=["bill_length_mm", "bill_depth_mm"])
print(f"all penguins: r = {d.bill_length_mm.corr(d.bill_depth_mm):.3f}  (n = {len(d)})")
for sp, grp in d.groupby("species"):
    print(f"{sp:<10} r = {grp.bill_length_mm.corr(grp.bill_depth_mm):.3f}  (n = {len(grp)})")
print(d.groupby("species")[["bill_length_mm", "bill_depth_mm"]].mean().round(1).to_string())

# Output:
# all penguins: r = -0.235  (n = 342)
# Adelie     r = 0.391  (n = 151)
# Chinstrap  r = 0.654  (n = 68)
# Gentoo     r = 0.643  (n = 123)
#            bill_length_mm  bill_depth_mm
# species
# Adelie               38.8           18.3
# Chinstrap            48.8           18.4
# Gentoo               47.5           15.0
```

**Why:** Inside every species, longer bills are also deeper (r about +0.4 to +0.65). Overall the sign flips to -0.235 because the species sit in different corners: Gentoo have long but shallow bills, Adelie short but deep bills. Pooling groups with different averages creates a trend that no group has. This is Simpson's paradox. Product version: a metric can rise in every country and still fall overall when the traffic mix shifts toward a low-metric country. Always ask 'is there a grouping variable?' before you trust a pooled correlation or a pooled rate.

**Say it to a PM:** Within each species, penguins with longer bills also have deeper bills. The overall negative number is an artefact of mixing three species with different bill shapes, so we should report the per-species relationship.

**Common mistakes:** Reading a pooled correlation as a within-group relationship. Treating correlation as causation. Forgetting that Pearson r measures only linear association and is sensitive to outliers (Spearman is the rank-based, robust version).

**Learn more:** [stats-descriptive](https://mahsa-agz.github.io/ds-handbook/#stats-descriptive), [stats-metric-drop](https://mahsa-agz.github.io/ds-handbook/#stats-metric-drop)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [AI and ML](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_01/4_ai.ipynb)